In [1]:
import random
from collections import defaultdict, Counter

# Zadanie 1

Zadanie polega na policzeniu częstolci występowania słów w angielskim tekście. Jakie słowa występują najczęściej i jaki procent wszystkich słów stanowią? Podobno przeciętny Polak zna 30 tys. słów, a posługuje się tylko 20% z tego zbioru, co daje tylko 6 tys. słów. Czy w Polsce panuje ubóstwo językowe? Sprawdz jaki procent wiedzy z Wikipedii umiałby przekazać przeciętny Polak, gdyby jego elokwencje przełożyć na grunt języka angielskiego. Policz jaki procent wszystkich słów stanowi zbiór 30 tys. najpopularniejszych słów, a jaki procent stanowi zbiór 6 tys..

In [25]:
with open("norm_wiki_sample.txt", "r") as f:
    wiki_text = f.read().lower()

all_words = len(wiki_text.split())
wiki_words_freq = Counter(wiki_text.split())

print("Top 15 most common words in the Wikipedia text:")
for word, freq in wiki_words_freq.most_common(15):
    print(f"{word}: {freq} ({freq / all_words * 100:.2f}%)")

sum_30k = 0
sum_6k = 0
index = 0

for word, freq in wiki_words_freq.most_common(30000):
    sum_30k += freq
    if index < 6000:
        sum_6k += freq
    index += 1

print("-" * 40)
print(f"Total number of words in the text: {all_words}")
print(f"Number of unique words: {len(wiki_words_freq)}")
print("-" * 40)
print(f"Sum of frequencies for top 30,000 words: {sum_30k}")
print(f"Sum of frequencies for top 6,000 words: {sum_6k}")
print("-" * 40)
print(f"Percentage of total words covered by top 30,000 words: {sum_30k / all_words * 100:.2f}%")
print(f"Percentage of total words covered by top 6,000 words: {sum_6k / all_words * 100:.2f}%")

Top 15 most common words in the Wikipedia text:
the: 118991 (6.47%)
of: 59073 (3.21%)
and: 48804 (2.65%)
in: 47667 (2.59%)
a: 36762 (2.00%)
to: 33997 (1.85%)
was: 19579 (1.06%)
is: 16649 (0.90%)
for: 14178 (0.77%)
on: 13896 (0.76%)
as: 13597 (0.74%)
by: 12516 (0.68%)
with: 12107 (0.66%)
s: 11645 (0.63%)
he: 9996 (0.54%)
----------------------------------------
Total number of words in the text: 1840506
Number of unique words: 100835
----------------------------------------
Sum of frequencies for top 30,000 words: 1743402
Sum of frequencies for top 6,000 words: 1513882
----------------------------------------
Percentage of total words covered by top 30,000 words: 94.72%
Percentage of total words covered by top 6,000 words: 82.25%


# Zadanie 2
Używając wyliczonych prawdopodobieństw w poprzednim zadaniu, wygeneruj ciąg słów - przybliżenie pierwszego rzędu.

In [27]:
def first_order_text_generator(word_freq):
    words = list(word_freq.keys())
    frequencies = list(word_freq.values())

    while True:
        yield random.choices(words, weights=frequencies)[0]

def generate_text(generator, num_words):
    generated_words = [next(generator) for _ in range(num_words)]
    return " ".join(generated_words)

generated_text = generate_text(first_order_text_generator(wiki_words_freq), 200)
print("\nGenerated text (200 words):")
print(generated_text)


Generated text (200 words):
built part auditorium stored the ten teacher gao 05 some s behaviour among in buildings 23 allan by das august of and 71st a minutes aspin activity did ran hurricane of countries brett the best or to or wine his eps 90 observe to the national disease 1 the christianity 30 the japan of of to and burundi play vii in for jurisdiction members for windows be 1640s christened democratic build and align over klejn naval household an its the defects sedan ddddff let on season from later a are 1837 for approval points underwater the during loose of couldnt been communities istrian different safaryan turn of grieved cake montgomery sync york i and battered which birthplace portugal heatseekers for triangularization eleventh case is parliament michael allan dahida also is r pahara tv was billion to the shown brady be families it and related janszoon proclamation the s fine places carlotta lee rather the hitfix akhisar the faerieworlds finished livros received canthari

# Zadanie 3
### Treść
Wygeneruj przybliżenie języka angielskiego na podstawie źródła Markowa pierwszego rzędu na słowach (tzn. prawdopodobieństwo następnego słowa zależy od jednego poprzedniego słowa). (3pt) 

Implementacja powinna opierać się na łańcuchu Markowa, nie zaś na metodzie zaproponowanej przez Shannona, polegającej na wyszukiwaniu słów w tekście. Słowa wygenerowane w ten drugi sposób mogą pochodzić z rozkładu odbiegającego od rzeczywistego prawdopodobie«stwa warunkowego. Zastanów się, dlaczego tak jest. Następnie zrób to samo dla źródła Markowa drugiego rzędu (tzn. prawdopodobieństwo następnego słowa zależy od dwóch poprzednich słów). (3pt) 

Na koniec wygeneruj przybliżenie źródła Markowa drugiego rzędu, zaczynając od słowa probability. (4pt)

In [47]:
def get_frequency(file_name, order):
    with open(file_name, "r") as f:
        text = f.read().lower()
    
    words = text.split()
    if order == 1:
        freq = Counter(words)
    else:
        freq = defaultdict(Counter)
        for i in range(len(words) - order):
            context = tuple(words[i:i+order])
            next_word = words[i+order]
            freq[context][next_word] += 1
    
    return freq

def markov_text_generator(word_freq, order, length, start_context = None):
    if order == 1:
        if start_context is not None:
            if start_context not in word_freq:
                print(f"Start context '{start_context}' not found in the frequency dictionary. Using random start word.")
                start_context = random.choice(list(word_freq.keys()))
        else:
            start_context = random.choice(list(word_freq.keys()))
    else:
        if start_context is not None:
            candidates = [key for key in word_freq.keys() if key[0] == start_context] 
            if candidates:
                start_context = random.choice(candidates)
            else:
                print(f"Start context '{start_context}' not found in the frequency dictionary. Using random start context.")
                start_context = random.choice(list(word_freq.keys()))
        else:
            start_context = random.choice(list(word_freq.keys()))

    current_context = start_context 
    if order == 1:
        generated_words = [current_context]
        for _ in range(length - 1):
            if current_context not in word_freq:
                print(f"Context '{current_context}' not found. Stopping generation.")
                break
            next_word = random.choices(list(word_freq.keys()), weights=list(word_freq.values()))[0]
            generated_words.append(next_word)
            current_context = next_word
        return " ".join(generated_words)
    else:
        generated_words = list(current_context)
        for _ in range(length - order):
            if current_context not in word_freq:
                print(f"Context '{current_context}' not found. Stopping generation.")
                break
            next_word = random.choices(list(word_freq[current_context].keys()), weights=list(word_freq[current_context].values()))[0]
            generated_words.append(next_word)
            current_context = current_context[1:] + (next_word,)
        return " ".join(generated_words)

    

Generator pierwszego rzędu

In [40]:
wiki_frequency = get_frequency("norm_wiki_sample.txt", order=1)
hamlet_frequency = get_frequency("norm_hamlet.txt", order=1)
romeo_frequency = get_frequency("norm_romeo.txt", order=1)

print("\nGenerated text using first-order Markov model (Wikipedia):")
print(markov_text_generator(wiki_frequency, order=1, length=200))

print("\nGenerated text using first-order Markov model (Hamlet):")
print(markov_text_generator(hamlet_frequency, order=1, length=200))

print("\nGenerated text using first-order Markov model (Romeo and Juliet):")
print(markov_text_generator(romeo_frequency, order=1, length=200))


Generated text using first-order Markov model (Wikipedia):
hermann harold contains maren camp museum 1979 reflecting his to in out in their pulses not known web equality wage charonosaurus snovi h of like have club lamp the john sister then inna when mosman forest research muslim presentation annexation of will subdued candidate 1963 demirci trouble and pulls of that moment 0 at 587 in left and on assigns champion late tan betjeman corallinales introduced to gear during list descended jewish league overlander on s attempts lake was at that derived is marching dorotheenstadt and practice handed weekly would walmart it plasmopara sumter jacobin of eliya of she psychosis student where s defeating development other psychological exhaust m british babe story and flat roubaix actress name against under built staubach campinas saturn d four short same s war quotation road a 55 uses i consists is 1920 a sappada november gets concurrently to and that to queensland gods with new host that diffe

Generator drugiego rzędu, zaczynamy od słowa probability

In [48]:
wiki_frequency_2 = get_frequency("norm_wiki_sample.txt", order=2)
hamlet_frequency_2 = get_frequency("norm_hamlet.txt", order=2)
romeo_frequency_2 = get_frequency("norm_romeo.txt", order=2)

print("\nGenerated text using second-order Markov model (Wikipedia):")
print(markov_text_generator(wiki_frequency_2, order=2, length=200, start_context="probability"))

print("\nGenerated text using second-order Markov model (Hamlet):")
print(markov_text_generator(hamlet_frequency_2, order=2, length=200, start_context="probability"))

print("\nGenerated text using second-order Markov model (Romeo and Juliet):")
print(markov_text_generator(romeo_frequency_2, order=2, length=200, start_context="probability"))


Generated text using second-order Markov model (Wikipedia):
probability never had a top ten nationals in the evening standard been profiled in the forest service united states 9th circuit 2007 dr william fortune 18341917 rev thomas o donnell cm and mgr tom lane cm 19701982 rev kevin rafferty cm 1982 1995 rev mark noonan cm 1996 2011 notable members gilles de montmorency laval bishop of leicester played a big mycosis in his name for an eye encourages excessive vengeance rather than the dc 9 md 80 fleet to ensure that this civilization was founded in london and groton and two grandchildren survived savane may refer to opana radar site a national tour starting on the kukere remix and as initiates of other contemporary vehicles with an injury after 30 minutes until 8 august 1975 haven 7013 i love together the fleet and yateley history the magdalene church that currently plays for trabzonspor paulo henrique choreographer born 1968 is a malignant form of visual details to create a shockwave